In [1]:
from __future__ import annotations
import os
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.config import MainConfig
from arc.model import KilatTransformer
from training.trainer import KilatTrainer, TrainingArguments
from data.collator import KilatDataCollator
from data.dataset import KilatDataset
from torchinfo import summary
import torch

# Use `config.yaml` To Make Your Life Easier

Instead of wrapping the config with `build_config` from `config.py`, you can use a YAML file directly.

**Why YAML is better:**

```python
# Old way: Hardcoded configs scattered in Python
config = KilatConfig(
    vocab_size=32000,
    n_embd=768,
    n_head=12,
    n_layer=12,
    ffn_mode="dense"
)

# Or worse: Wrapped with build_config boilerplate
config = build_config(
    model_size="small",
    use_moe=False,
    batch_size=32
)

# Better: Clean YAML configuration
import yaml

with open("config.yaml", "r") as f:
    config_dict = yaml.safe_load(f)
    config = KilatConfig(**config_dict)
```

**Example `config.yaml`:**
```yaml
# Model architecture
vocab_size: 32000
n_embd: 768
n_head: 12
n_layer: 12
ffn_mode: "dense"

# Training settings
batch_size: 32
learning_rate: 3e-4
warmup_steps: 2000

# System
dtype: "bfloat16"
gradient_checkpointing: true
```

**Benefits:**
- **Separation of concerns** - Code vs configuration
- **Easy to version control** - Human-readable diffs
- **Experiment tracking** - Just copy the YAML file
- **No code changes** to tweak hyperparameters
- **Multi-environment support** - dev.yaml, prod.yaml, benchmark.yaml

**Quick tip:** Use Pydantic or dataclasses to validate YAML at load time:
```python
class KilatConfig:
    @classmethod
    def from_yaml(cls, path: str):
        with open(path) as f:
            data = yaml.safe_load(f)
        return cls(**data)  # Validation happens here
```

---

This keeps your code clean and your experiments reproducible! 

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
config = MainConfig.from_yaml("../configs/small_dense.yaml")
model = KilatTransformer(config.model)
model = model.to(device)

In [5]:
stats = summary(
    model,
    input_data={
        'input_ids': torch.randint(0, 32000, (1, 512), dtype=torch.long, device=device)
    },
    device=device, 
    depth=3,
    col_names=["input_size", "output_size", "num_params", "mult_adds"],
    verbose=1
)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Mult-Adds
KilatTransformer                         --                        [1, 512, 32000]           --                        --
├─Embedding: 1-1                         [1, 512]                  [1, 512, 768]             24,576,000                24,576,000
├─Dropout: 1-2                           [1, 512, 768]             [1, 512, 768]             --                        --
├─ModuleList: 1-3                        --                        --                        --                        --
│    └─Block: 2-1                        [1, 512, 768]             [1, 512, 768]             --                        --
│    │    └─RMSNorm: 3-1                 [1, 512, 768]             [1, 512, 768]             768                       768
│    │    └─KilatAttention: 3-2          [1, 512, 768]             [1, 512, 768]             3,172,614                 3,172,608
│

In [6]:
total_macs = stats.total_mult_adds  
flops_strict = 2 * total_macs
flops_loose = total_macs
print(f"Total MACs        : {total_macs/1e6:.2f} M")
print(f"FLOPs (strict, 2× MACs) : {flops_strict/1e6:.2f} MFLOPs")
print(f"FLOPs (loose, 1× MACs) : {flops_loose/1e6:.2f} MFLOPs")

Total MACs        : 143.87 M
FLOPs (strict, 2× MACs) : 287.73 MFLOPs
FLOPs (loose, 1× MACs) : 143.87 MFLOPs
